# RAG — Voice of Customer com Olist

Notebook principal da entrega. Executa a jornada completa sobre as tabelas reais do Olist: validação dos dados, enriquecimento relacional, construção do corpus, embeddings, FAISS, recuperação semântica, geração fundamentada e testes de cenários sem evidência.


## 0. Ambiente e imports

Se `python-dotenv`, `faiss-cpu`, `sentence-transformers` ou `openai` não estiverem instalados, execute a instalação abaixo no mesmo kernel selecionado pelo VS Code. Depois reinicie o kernel.


In [ ]:
import sys
from pathlib import Path
REQ_FILE = Path.cwd() / "requirements.txt"
if not REQ_FILE.exists():
    REQ_FILE = Path.cwd().parent / "requirements.txt"
print("Requirements:", REQ_FILE)
!{sys.executable} -m pip install -r "{REQ_FILE}"


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config.settings import settings
from src.data.loader import load_olist_raw
from src.data.preprocessing import enrich_reviews, build_documents
from src.embeddings.embedder import EmbeddingModel
from src.retrieval.retriever import VectorRetriever
from src.rag.pipeline import RAGPipeline
from src.generation.llm import LLMGenerator

RAW_DIR = settings.data_dir / "raw"
INDEX_DIR = settings.data_dir / "vectorstore"
print("Projeto:", ROOT)
print("RAW:", RAW_DIR)


## 1. Validação das tabelas RAW

O corpus textual é formado pelas avaliações. As demais tabelas enriquecem cada avaliação por `order_id`, `customer_id`, `product_id` e `seller_id`.


In [ ]:
expected = [
    "olist_customers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_orders_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "product_category_name_translation.csv",
]
status = pd.DataFrame({"arquivo": expected, "existe": [(RAW_DIR / f).exists() for f in expected]})
display(status)
missing = status.loc[~status["existe"], "arquivo"].tolist()
if missing:
    raise FileNotFoundError("Arquivos ausentes em data/raw: " + ", ".join(missing))


In [ ]:
raw = load_olist_raw(RAW_DIR)
summary = pd.DataFrame([{"tabela": k, "linhas": len(v), "colunas": len(v.columns)} for k, v in raw.items()])
display(summary)


## 2. Construção da base de conhecimento

Uma avaliação permanece uma única unidade semântica. Itens, categorias, vendedores e pagamentos são agregados por pedido antes do enriquecimento, evitando duplicação de comentários no índice.


In [ ]:
reviews = raw["reviews"]
print("Reviews totais:", len(reviews))
print("Review IDs distintos:", reviews["review_id"].nunique())
print("Comentários preenchidos:", reviews["review_comment_message"].fillna("").astype(str).str.strip().ne("").sum())

prepared = enrich_reviews(raw)
documents = build_documents(prepared)
print("Documentos finais:", len(documents))
print("Exemplo de documento:")
display(pd.DataFrame([documents[0]]))


In [ ]:
metadata_df = pd.DataFrame([d["metadata"] for d in documents])
metadata_cols = [c for c in ["review_score","customer_state","product_category_name_english","delivery_days","delivery_delay_days","payment_type"] if c in metadata_df.columns]
display(metadata_df[metadata_cols].head())


## 3. Embeddings e índice vetorial

Modelo: `paraphrase-multilingual-MiniLM-L12-v2`. Os vetores são normalizados e indexados no FAISS com similaridade por produto interno, equivalente a cosine similarity após normalização.


In [ ]:
embedder = EmbeddingModel(settings.embedding_model)
texts = [d["text"] for d in documents]
embeddings = embedder.encode(texts)
retriever = VectorRetriever(embeddings, documents)
retriever.save(INDEX_DIR)
print("Modelo:", settings.embedding_model)
print("Dimensão:", embeddings.shape[1])
print("Vetores indexados:", retriever.size)


## 4. Recuperação semântica


In [ ]:
question = "Quais são os principais problemas relatados pelos clientes sobre a entrega?"
evidence = retriever.search(embedder.encode([question])[0], top_k=settings.top_k, min_score=settings.min_relevance_score)
print("Pergunta:", question)
print("Evidências recuperadas:", len(evidence))
for item in evidence:
    print(f"[{item['document_id']}] score={item['score']:.3f} | nota={item.get('metadata',{}).get('review_score')} | {item['text']}")


## 5. Geração RAG fundamentada

A geração recebe somente as evidências recuperadas. O prompt instrui o modelo a não inventar fatos e a declarar insuficiência quando a base não sustenta a resposta.


In [ ]:
generator = LLMGenerator(settings.llm_model)
pipeline = RAGPipeline(embedder, retriever, generator, settings.top_k, settings.min_relevance_score)
result = pipeline.ask(question)
print(result["answer"])
print("\nIDs das evidências:", [x["document_id"] for x in result["evidence"]])


## 6. Cenário sem evidência / fora do escopo

Este teste verifica o comportamento quando a recuperação não encontra evidência acima do limiar configurado. O resultado esperado é uma resposta de insuficiência, sem fabricação de informação.


In [ ]:
out_of_scope = "Qual é a previsão do tempo para amanhã?"
result_out = pipeline.ask(out_of_scope)
print(result_out["answer"])
print("Tem evidência:", result_out["has_evidence"])


## 7. Exemplos de consultas sobre a Voz do Cliente


In [ ]:
questions = [
    "Quais são os principais problemas relacionados à entrega?",
    "Quais reclamações aparecem nas avaliações de baixa nota?",
    "Quais categorias de produtos aparecem nas avaliações recuperadas?",
]
for q in questions:
    ev = pipeline.retrieve(q)
    print("\nPERGUNTA:", q)
    print("EVIDÊNCIAS:", len(ev), [x["document_id"] for x in ev])


## 8. Persistência e reprodução

O índice produzido pelo notebook é salvo em `data/vectorstore/`. A mesma construção pode ser reproduzida pelo script `python scripts/build_index.py`. Para consultas no terminal: `python scripts/query_rag.py "sua pergunta"`. Para a interface: `streamlit run app/streamlit_app.py`.


In [ ]:
loaded = VectorRetriever.load(INDEX_DIR)
print("Índice persistido e recarregado com sucesso.")
print("Vetores:", loaded.size)
print("Documentos persistidos:", len(loaded.documents))
assert loaded.size == len(documents)


## 9. Conclusão técnica

O fluxo implementado é: **pergunta → embedding → recuperação semântica → contexto de evidências → LLM → resposta fundamentada**. Cada documento mantém `document_id` e metadados relacionais para rastreabilidade. A ausência de evidência acima do limiar impede que o pipeline entregue uma resposta baseada em contexto inexistente.
